[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helvecioneto/datascience/blob/main/praticas/pratica02.ipynb)

# Aula Prática 02 — Pipeline CRISP-DM e Análise Exploratória de Dados

**Disciplina:** Introdução à Ciência de Dados (NSA A110) · **Professor:** Helvecio Bezerra Leal Neto · UFOPA
**Data:** 17/09/2026, 19h às 22h · **Modalidade:** prática em equipes (2 a 3 pessoas)

**Objetivos de aprendizagem**
- Enquadrar uma pergunta de negócio e traduzi-la em perguntas que os dados respondem (Fase 1 do CRISP-DM).
- Construir um dicionário de dados e diagnosticar faltantes, duplicatas, tipos inconsistentes e outliers (Fase 2).
- Tratar cada problema com pandas, registrando por escrito a justificativa de cada decisão (Fase 3).
- Produzir uma EDA interpretável: distribuições, boxplots por grupo, correlações e dispersões, com uma frase de leitura por gráfico.
- Reconhecer as armadilhas de Grus: correlação não é causalidade, viés de amostragem e Paradoxo de Simpson.

**Pré-requisitos:** Aula 05 (CRISP-DM, reprodutibilidade, notebook auditável), Prática 01 (pandas básico) e noções de estatística descritiva.

**Fontes:** Kelleher & Tierney (2018) cap. 3 [pág. 40-42]; Grus (2021) cap. 5 e 10; Cady (2017) cap. 2 e 5.

## 0. Preparação do ambiente

Instalamos só o que é consolidado no ecossistema: pandas (tabelas), numpy (números), matplotlib e seaborn (gráficos). O seaborn também traz os datasets públicos que usaremos.

In [ ]:
# pandas: manipulação de tabelas | numpy: geração de números e semente
# matplotlib: base de todos os gráficos | seaborn: gráficos estatísticos + datasets públicos
%pip install pandas numpy matplotlib seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Semente fixa: tudo o que envolve aleatoriedade neste notebook será reproduzível (Aula 05)
SEED = 42
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

print("pandas", pd.__version__, "| seaborn", sns.__version__, "| semente:", SEED)

## Fase 1 — Entendimento do negócio

O CRISP-DM começa no negócio, não nos dados [Kelleher 2018 · pág. 41-42]. Antes de abrir qualquer tabela, preenchemos a **ficha do projeto**: quem pergunta, o que quer decidir e como saberemos se deu certo.

In [ ]:
# A ficha do projeto é a primeira coisa que um auditor lê. Sem ela, "não sabemos o que perguntar aos dados".
ficha_projeto = {
    "Cliente (quem pergunta)":   "Companhia de navegação White Star Line (cenário histórico do Titanic)",
    "Problema de negócio":       "Entender quais fatores estiveram associados à sobrevivência no naufrágio de 1912",
    "Decisão que será apoiada":  "Priorizar políticas de segurança: distribuição de botes, ordem de evacuação, acesso ao convés",
    "Pergunta central":          "Que características dos passageiros mais se relacionam com sobreviver?",
    "Critério de sucesso":       "Uma EDA reprodutível que aponte 3 fatores com evidência clara nos dados",
    "Fora do escopo":            "Prever sobrevivência com aprendizado de máquina (isso é a Fase 4, Modelagem)",
}

for campo, valor in ficha_projeto.items():
    print(f"{campo:28s}: {valor}")

In [ ]:
# Toda pergunta de negócio precisa virar uma pergunta que uma TABELA responde.
# Repare: a coluna "métrica" já diz qual conta faremos na EDA.
perguntas = pd.DataFrame([
    ("Mulheres e crianças tiveram prioridade?",   "Taxa de sobrevivência por sexo e por faixa etária", "sobreviveu, sexo, idade"),
    ("A classe da passagem fez diferença?",       "Taxa de sobrevivência por classe",                  "sobreviveu, classe"),
    ("Quem pagou mais sobreviveu mais?",          "Correlação tarifa × sobrevivência, dentro de cada classe", "tarifa, classe, sobreviveu"),
    ("Viajar em família ajudou ou atrapalhou?",   "Taxa de sobrevivência por tamanho da família",      "irmaos_conjuge, pais_filhos"),
], columns=["pergunta_de_negocio", "metrica_nos_dados", "colunas_necessarias"])

perguntas

## Fase 2 — Entendimento dos dados: primeiro contato

Carregamos o dataset público `titanic` do seaborn e, de propósito, o **sujamos** de forma controlada (duplicatas, tipos trocados, categorias inconsistentes, erros de digitação). Assim conhecemos o gabarito e podemos conferir se a limpeza funcionou.

In [ ]:
# Dataset público embutido no seaborn (891 passageiros). Selecionamos e renomeamos as colunas para PT-BR.
colunas = {
    "survived": "sobreviveu", "pclass": "classe", "sex": "sexo", "age": "idade",
    "sibsp": "irmaos_conjuge", "parch": "pais_filhos", "fare": "tarifa",
    "embarked": "embarque", "deck": "conves",
}
bruto = sns.load_dataset("titanic")[list(colunas)].rename(columns=colunas)

print("Formato original (linhas, colunas):", bruto.shape)
bruto.head()

In [ ]:
def sujar(df, rng):
    """Injeta problemas típicos de dados reais. Em um projeto de verdade os dados JÁ chegam assim."""
    df = df.copy()
    # 1) Duplicatas: 25 linhas copiadas (ex.: dois cadastros do mesmo passageiro)
    df = pd.concat([df, df.sample(25, random_state=SEED)], ignore_index=True)
    # 2) Categorias inconsistentes: 'male' vira 'MALE' e ' male ' (digitação humana)
    idx = rng.choice(df.index, 60, replace=False)
    df.loc[idx[:30], "sexo"] = df.loc[idx[:30], "sexo"].str.upper()
    df.loc[idx[30:], "sexo"] = " " + df.loc[idx[30:], "sexo"] + " "
    # 3) Tipo inconsistente: 200 tarifas gravadas como texto com vírgula decimal ("7,25")
    idx = rng.choice(df.index, 200, replace=False)
    df["tarifa"] = df["tarifa"].astype(object)
    df.loc[idx, "tarifa"] = df.loc[idx, "tarifa"].map(lambda v: f"{v:.2f}".replace(".", ","))
    # 4) Erros de digitação: idade 999 e tarifa negativa
    idx = rng.choice(df.index[df["idade"].notna()], 3, replace=False)
    df.loc[idx, "idade"] = 999
    idx = rng.choice(df.index, 2, replace=False)
    df.loc[idx, "tarifa"] = -50.0
    # Embaralha as linhas para que as duplicatas não fiquem no fim
    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df_sujo = sujar(bruto, rng)
print("Formato do dataset sujo:", df_sujo.shape, "(o bruto tinha", bruto.shape[0], "linhas)")
df_sujo.head(8)

### 2.1 Dicionário de dados

O dicionário registra o que cada coluna **deveria** ser. Comparar o tipo esperado com o tipo real é o jeito mais rápido de achar problemas [Cady 2017 · pág. 30-31].

In [ ]:
dicionario = pd.DataFrame([
    ("sobreviveu",     "int",      "0 = não sobreviveu, 1 = sobreviveu",          "alvo"),
    ("classe",         "int",      "Classe da passagem: 1ª, 2ª ou 3ª",             "atributo"),
    ("sexo",           "category", "female / male",                                "atributo"),
    ("idade",          "float",    "Idade em anos (fração para bebês)",            "atributo"),
    ("irmaos_conjuge", "int",      "Nº de irmãos + cônjuge a bordo",               "atributo"),
    ("pais_filhos",    "int",      "Nº de pais + filhos a bordo",                  "atributo"),
    ("tarifa",         "float",    "Valor pago pela passagem, em libras de 1912",  "atributo"),
    ("embarque",       "category", "Porto: C = Cherbourg, Q = Queenstown, S = Southampton", "atributo"),
    ("conves",         "category", "Letra do convés da cabine (A a G)",            "atributo"),
], columns=["coluna", "tipo_esperado", "significado", "papel"]).set_index("coluna")

# Cruzamos o esperado com o real: onde os dois divergem há trabalho de preparação
dicionario["tipo_real"] = df_sujo.dtypes.astype(str)
dicionario["divergente"] = dicionario["tipo_esperado"] != dicionario["tipo_real"].str.replace("64", "")
dicionario

In [ ]:
# O info() é o raio-X: quantos valores não nulos por coluna e o tipo que o pandas inferiu.
# REPARE: 'tarifa' aparece como object (texto), sinal de que há números gravados como string.
df_sujo.info()

### 2.2 Diagnóstico de qualidade

Quatro perguntas, quatro linhas de código: quantos faltam, quantas duplicatas, quais categorias existem e se os números fazem sentido.

In [ ]:
# Faltantes: contagem e percentual por coluna (o percentual é o que decide remover ou imputar)
faltantes = pd.DataFrame({
    "n_faltantes": df_sujo.isna().sum(),
    "pct_faltantes": (df_sujo.isna().mean() * 100).round(1),
}).sort_values("pct_faltantes", ascending=False)

print("Duplicatas exatas encontradas:", df_sujo.duplicated().sum())
faltantes

In [ ]:
# Categorias: value_counts revela variantes que deveriam ser a mesma coisa
print("Valores distintos em 'sexo':")
print(df_sujo["sexo"].value_counts(dropna=False), "\n")

print("Valores distintos em 'embarque':")
print(df_sujo["embarque"].value_counts(dropna=False))

In [ ]:
# Os faltantes são aleatórios ou têm padrão? Se dependem de outra coluna, remover cria VIÉS.
falta_conves = (df_sujo.assign(conves_faltante=df_sujo["conves"].isna())
                        .groupby("classe")["conves_faltante"].mean().mul(100).round(1))

print("% de convés faltante por classe:")
print(falta_conves)
print("\nInterpretação: o convés falta quase sempre na 3ª classe. Não é acaso, é o registro histórico"
      " que priorizava passageiros ricos. Qualquer análise por convés falaria só da 1ª classe.")

In [ ]:
# Visão gráfica dos faltantes: cada linha branca é um valor ausente
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(df_sujo.isna().T, cbar=False, cmap="Greys", ax=ax)
ax.set_title("Mapa de valores faltantes (branco = ausente)")
ax.set_xlabel("índice da linha")
plt.tight_layout(); plt.show()

print("O que observar: 'conves' é quase todo branco (77%), 'idade' tem falhas espalhadas (~20%),"
      " 'embarque' tem 2 buracos. Cada caso pede uma decisão diferente.")

## Fase 3 — Preparação dos dados

É a fase mais demorada de um projeto real. Cada decisão vai para um **registro de decisões**, no espírito de mensagem de commit: problema, o que fizemos e por quê. Sem isso o notebook não é auditável.

In [ ]:
# Registro de decisões: a lista que vira a seção "Preparação" do relatório da equipe
decisoes = []

def registrar(problema, decisao, justificativa, linhas_afetadas):
    decisoes.append({"problema": problema, "decisao": decisao,
                     "justificativa": justificativa, "linhas_afetadas": linhas_afetadas})
    print(f"📝 {problema} → {decisao} ({linhas_afetadas} linhas)")

# Cópia de trabalho: o bruto sujo permanece IMUTÁVEL para podermos voltar a ele
df = df_sujo.copy()

### 3.1 Tipos inconsistentes

Primeiro corrigimos os tipos: antes disso nem `describe()` nem a busca de outliers funcionam para `tarifa`.

In [ ]:
# Caso que dá errado de propósito: converter direto com astype falha na primeira vírgula
try:
    df["tarifa"].astype(float)
except ValueError as erro:
    print("astype(float) falhou:", erro)
    print("Motivo: '7,25' não é um número para o Python. Precisamos padronizar o separador antes.")

In [ ]:
# Correção: texto → troca vírgula por ponto → to_numeric (errors='coerce' transforma o que sobrar em NaN)
tarifa_texto = df["tarifa"].astype(str).str.replace(",", ".", regex=False)
df["tarifa"] = pd.to_numeric(tarifa_texto, errors="coerce")

registrar("tarifa gravada como texto com vírgula decimal",
          "converter para float trocando ',' por '.'",
          "valores idênticos aos numéricos, só o formato de digitação diferia", 200)

print("\nTipo agora:", df["tarifa"].dtype, "| NaN gerados pela conversão:", df["tarifa"].isna().sum())

### 3.2 Categorias inconsistentes

In [ ]:
# strip() remove espaços nas pontas, lower() padroniza a caixa. Fazemos ANTES de tratar duplicatas.
n_variantes_antes = df["sexo"].nunique()
df["sexo"] = df["sexo"].str.strip().str.lower()

registrar("'sexo' com variantes de caixa e espaços",
          "padronizar com strip() + lower()",
          f"{n_variantes_antes} variantes representavam apenas 2 categorias reais", 60)

print("\nDepois da padronização:")
print(df["sexo"].value_counts())

### 3.3 Duplicatas

Repare na ordem: se tivéssemos removido duplicatas antes de padronizar `sexo` e `tarifa`, várias cópias teriam escapado por diferirem só no formato.

In [ ]:
# Comparação: quantas duplicatas eram visíveis no bruto sujo vs agora, após padronizar
print("Duplicatas visíveis ANTES de padronizar:", df_sujo.duplicated().sum())
print("Duplicatas visíveis DEPOIS de padronizar:", df.duplicated().sum())
print("Lição: padronizar primeiro revela duplicatas escondidas por formatação.")

In [ ]:
# Caso que dá errado de propósito: drop_duplicates() SEM atribuição não altera o DataFrame
n_antes = len(df)
df.drop_duplicates()
print("Sem atribuição, o tamanho continua igual?", len(df) == n_antes)

In [ ]:
# Jeito certo: atribuir o resultado (ou usar inplace=True) e reindexar
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

registrar("linhas exatamente iguais",
          "remover, mantendo a primeira ocorrência",
          "sem ID de passageiro, linhas idênticas em 9 colunas são tratadas como o mesmo registro (risco documentado)", n_dup)

print("\nLinhas restantes:", len(df))

### 3.4 Outliers: erro de digitação ou valor real?

Outlier não é sinônimo de erro. A regra: valores **impossíveis** (idade 999, tarifa negativa) viram faltante; valores **improváveis mas possíveis** ficam, com justificativa.

In [ ]:
# describe() mostra os extremos: repare no max de idade e no min de tarifa
df[["idade", "tarifa"]].describe().T.round(2)

In [ ]:
# Caso que dá errado de propósito: imputar a média ANTES de tratar os impossíveis
media_com_999 = df["idade"].mean()
media_sem_999 = df.loc[df["idade"] <= 100, "idade"].mean()

print(f"Média da idade COM os 999: {media_com_999:.1f} anos")
print(f"Média da idade SEM os 999: {media_sem_999:.1f} anos")
print("Se tivéssemos imputado pela média agora, teríamos envelhecido 177 passageiros por causa de 3 erros de digitação.")

In [ ]:
# Valores impossíveis → NaN (serão imputados na etapa seguinte, junto com os faltantes originais)
impossiveis_idade = df["idade"] > 100
impossiveis_tarifa = df["tarifa"] < 0
print("Linhas suspeitas:")
print(df.loc[impossiveis_idade | impossiveis_tarifa, ["classe", "sexo", "idade", "tarifa"]])

df.loc[impossiveis_idade, "idade"] = np.nan
df.loc[impossiveis_tarifa, "tarifa"] = np.nan

registrar("idade > 100 anos", "substituir por NaN", "ninguém a bordo tinha 999 anos: erro de digitação", int(impossiveis_idade.sum()))
registrar("tarifa negativa", "substituir por NaN", "preço de passagem não pode ser negativo: erro de registro", int(impossiveis_tarifa.sum()))

In [ ]:
# Regra do IQR para a tarifa: acima de Q3 + 1,5·IQR é "outlier estatístico". Mas é erro?
q1, q3 = df["tarifa"].quantile([0.25, 0.75])
limite = q3 + 1.5 * (q3 - q1)
acima = df[df["tarifa"] > limite]

print(f"Limite superior do IQR: {limite:.1f} libras | passageiros acima: {len(acima)}")
print("Classe desses passageiros:")
print(acima["classe"].value_counts())

registrar("tarifas acima do limite IQR", "MANTER",
          "quase todas são de 1ª classe: valores altos são reais, não erros. Remover apagaria justamente os ricos", len(acima))

### 3.5 Faltantes: remover ou imputar?

Três colunas, três decisões diferentes. O critério: quantos faltam e se a falta tem padrão (viés). Marcamos o que foi imputado para poder auditar depois.

In [ ]:
# 'conves': 77% faltante e concentrado na 3ª classe → imputar seria inventar; removemos a coluna
pct_conves = df["conves"].isna().mean() * 100
df = df.drop(columns=["conves"])
registrar("'conves' com 77% de faltantes", "remover a coluna",
          "imputar inventaria dados para a 3ª classe inteira; a informação não é recuperável", int(pct_conves))

# 'embarque': 2 faltantes em ~890 → moda (valor mais frequente), efeito desprezível
moda = df["embarque"].mode()[0]
n_emb = df["embarque"].isna().sum()
df["embarque"] = df["embarque"].fillna(moda)
registrar("'embarque' com 2 faltantes", f"imputar pela moda ('{moda}')", "2 linhas não alteram nenhuma taxa", int(n_emb))

In [ ]:
# 'idade': ~20% faltante. Mediana por (classe, sexo) respeita as diferenças entre grupos.
# transform() devolve a mediana do grupo alinhada linha a linha, pronta para o fillna.
df["idade_imputada"] = df["idade"].isna()          # flag de auditoria
mediana_grupo = df.groupby(["classe", "sexo"])["idade"].transform("median")
df["idade"] = df["idade"].fillna(mediana_grupo)

registrar("'idade' com ~20% de faltantes", "imputar pela mediana de (classe, sexo) + coluna-flag",
          "a mediana resiste a extremos; por grupo, porque a 1ª classe é bem mais velha que a 3ª", int(df["idade_imputada"].sum()))

print("\nMedianas usadas por grupo:")
print(df.groupby(["classe", "sexo"])["idade"].median().unstack().round(1))

In [ ]:
# Efeito visual da imputação: mediana GLOBAL cria um pico artificial; por grupo, o pico se espalha
idade_original = df.loc[~df["idade_imputada"], "idade"]
imputacao_global = df["idade"].where(~df["idade_imputada"], idade_original.median())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, serie, titulo in zip(axes, [idade_original, imputacao_global, df["idade"]],
                             ["Só valores originais", "Imputada pela mediana global", "Imputada por (classe, sexo)"]):
    ax.hist(serie, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(titulo); ax.set_xlabel("idade")
plt.tight_layout(); plt.show()

print("O que observar: a mediana global empilha 177 pessoas na mesma barra (~28 anos); por grupo o efeito fica diluído.")

In [ ]:
# Tarifa: os 2 NaN que criamos no passo 3.4 → mediana da classe
df["tarifa"] = df["tarifa"].fillna(df.groupby("classe")["tarifa"].transform("median"))

# Tipos finais coerentes com o dicionário
df["sexo"] = df["sexo"].astype("category")
df["embarque"] = df["embarque"].astype("category")

print("Faltantes restantes por coluna:")
print(df.isna().sum())
print("\nFormato final:", df.shape)

In [ ]:
# O registro de decisões é o produto MAIS importante desta fase. Ele vai no relatório da equipe.
pd.DataFrame(decisoes)

## Fase 2 de novo — Análise exploratória nos dados limpos

O CRISP-DM é iterativo: voltamos ao entendimento dos dados, agora com dados confiáveis [Kelleher 2018 · pág. 41-42]. Regra da aula: **cada gráfico termina com uma frase de interpretação** no `print()`.

### 4.1 Distribuições univariadas

In [ ]:
# Histogramas de idade e tarifa. A tarifa é muito assimétrica: poucos pagaram fortunas.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df["idade"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Idade"); axes[0].set_xlabel("anos")
axes[1].hist(df["tarifa"], bins=40, color="darkorange", edgecolor="white")
axes[1].set_title("Tarifa (escala linear)"); axes[1].set_xlabel("libras")
plt.tight_layout(); plt.show()

print(f"Interpretação: idade concentrada entre 20 e 40 anos (mediana {df['idade'].median():.0f}); "
      f"tarifa com cauda longa: 75% pagaram até {df['tarifa'].quantile(.75):.0f} libras, mas o máximo é {df['tarifa'].max():.0f}.")

In [ ]:
# Escala logarítmica (Cady 2017, cap. 5): revela a estrutura de uma variável com cauda longa
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(df.loc[df["tarifa"] > 0, "tarifa"], bins=np.logspace(0, 3, 40), color="darkorange", edgecolor="white")
ax.set_xscale("log")
ax.set_title("Tarifa em escala log"); ax.set_xlabel("libras (log)")
plt.tight_layout(); plt.show()

print("Interpretação: em log aparecem 3 'morros', que correspondem aproximadamente às 3 classes de passagem.")

### 4.2 Boxplots por grupo: comparar distribuições, não só médias

In [ ]:
# Boxplot de tarifa por classe: a mediana (linha) e a dispersão (caixa) de cada grupo lado a lado
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.boxplot(data=df, x="classe", y="tarifa", ax=axes[0], palette="Set2", hue="classe", legend=False)
axes[0].set_yscale("log"); axes[0].set_title("Tarifa por classe (log)")

sns.boxplot(data=df, x="sobreviveu", y="idade", hue="sexo", ax=axes[1], palette="Set1")
axes[1].set_title("Idade por sobrevivência e sexo")
plt.tight_layout(); plt.show()

print("Interpretação: as tarifas quase não se sobrepõem entre classes (bom indicador do grupo social);")
print("a idade dos que sobreviveram é parecida com a dos que não sobreviveram, exceto pelas crianças.")

In [ ]:
# Taxa de sobrevivência cruzando classe × sexo: a tabela que responde à pergunta central do negócio
taxa = df.pivot_table(index="classe", columns="sexo", values="sobreviveu", aggfunc="mean", observed=True).round(2)
contagem = df.pivot_table(index="classe", columns="sexo", values="sobreviveu", aggfunc="size", observed=True)

print("Taxa de sobrevivência (proporção):")
print(taxa)
print("\nTamanho de cada grupo (sempre olhe o n antes de confiar numa taxa):")
print(contagem)
print(f"\nInterpretação: mulheres da 1ª classe sobreviveram em {taxa.loc[1,'female']:.0%}; homens da 3ª, em {taxa.loc[3,'male']:.0%}.")

In [ ]:
# O mesmo cruzamento como gráfico de barras: a leitura fica imediata
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(data=df, x="classe", y="sobreviveu", hue="sexo", errorbar=("ci", 95), palette="Set1", ax=ax)
ax.set_ylabel("taxa de sobrevivência"); ax.set_title("Sobrevivência por classe e sexo (barras de erro = IC 95%)")
plt.tight_layout(); plt.show()

print("Interpretação: sexo pesa mais que classe; dentro de cada sexo, a classe ainda separa os grupos.")

### 4.3 Matriz de correlação: Pearson e Spearman

Pearson mede relação linear; Spearman mede relação monotônica por postos e resiste a caudas longas como a da tarifa [Cady 2017 · pág. 117-118].

In [ ]:
# Selecionamos só colunas numéricas. Repare que 'sobreviveu' (0/1) entra: correlação com um binário é válida
numericas = df[["sobreviveu", "classe", "idade", "irmaos_conjuge", "pais_filhos", "tarifa"]]
pearson = numericas.corr(method="pearson")
spearman = numericas.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, matriz, nome in zip(axes, [pearson, spearman], ["Pearson", "Spearman"]):
    sns.heatmap(matriz, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
    ax.set_title(f"Correlação de {nome}")
plt.tight_layout(); plt.show()

print("Interpretação: tarifa ↔ classe é a relação mais forte (negativa: classe 1 paga mais).")
print(f"tarifa ↔ sobreviveu: Pearson {pearson.loc['tarifa','sobreviveu']:.2f} vs Spearman {spearman.loc['tarifa','sobreviveu']:.2f}."
      " Spearman é maior porque não é distorcido pelas tarifas gigantes.")

In [ ]:
# Dispersão idade × tarifa, colorida por sobrevivência. Escala log de novo na tarifa.
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=df, x="idade", y="tarifa", hue="sobreviveu", style="classe", alpha=0.6, palette="Set1", ax=ax)
ax.set_yscale("log"); ax.set_title("Idade × tarifa (cor = sobreviveu, marcador = classe)")
plt.tight_layout(); plt.show()

print("Interpretação: não há relação clara entre idade e tarifa; a cor se concentra em cima (tarifas altas),"
      " o que reforça a leitura por classe, e não por idade.")

### 4.4 Alerta 1 — Correlação não é causalidade

Tarifa e sobrevivência estão correlacionadas. Pagar mais **causou** sobreviver? Testamos a hipótese alternativa: a tarifa só é um espelho da classe.

In [ ]:
# Se a tarifa tivesse efeito próprio, a correlação persistiria DENTRO de cada classe
print(f"Correlação geral tarifa × sobreviveu: {df['tarifa'].corr(df['sobreviveu']):.2f}\n")
for classe, grupo in df.groupby("classe"):
    print(f"  dentro da classe {classe}: {grupo['tarifa'].corr(grupo['sobreviveu']):.2f}  (n = {len(grupo)})")

print("\nInterpretação: dentro de cada classe a correlação encolhe muito. A tarifa não 'salvou' ninguém;")
print("ela apenas indica a classe, que dava acesso ao convés dos botes. A classe é a variável de confusão.")

### 4.5 Alerta 2 — Viés de amostragem: quem está (e quem não está) nos dados

Os faltantes não são aleatórios. Verificamos se a idade faltava mais para certos grupos: se sim, qualquer análise "só com quem tem idade" seria enviesada.

In [ ]:
# A flag 'idade_imputada' guardou quem não tinha idade registrada. Cruzamos com classe e sobrevivência.
vies = df.pivot_table(index="classe", columns="sobreviveu", values="idade_imputada", aggfunc="mean", observed=True).mul(100).round(1)
print("% de idade originalmente faltante por classe e sobrevivência:")
print(vies)

print("\nInterpretação: a idade falta muito mais na 3ª classe e entre quem NÃO sobreviveu.")
print("Se tivéssemos removido as linhas sem idade (dropna), teríamos apagado justamente os mais vulneráveis")
print("e a taxa de sobrevivência 'calculada' subiria artificialmente. Foi por isso que imputamos.")

In [ ]:
# Prova numérica do viés: taxa de sobrevivência com e sem as linhas que tinham idade faltante
taxa_todos = df["sobreviveu"].mean()
taxa_dropna = df.loc[~df["idade_imputada"], "sobreviveu"].mean()

print(f"Taxa de sobrevivência com TODOS os passageiros: {taxa_todos:.1%}")
print(f"Taxa se tivéssemos usado dropna() na idade:      {taxa_dropna:.1%}")
print(f"Diferença: {abs(taxa_dropna - taxa_todos) * 100:.1f} pontos percentuais só pela escolha de tratamento dos faltantes.")

### 4.6 Alerta 3 — Paradoxo de Simpson

Uma correlação pode **inverter o sinal** quando separamos os dados por grupo (Grus, cap. 5). O dataset público `penguins` tem o exemplo mais famoso.

In [ ]:
# Dataset público do seaborn: medidas de 3 espécies de pinguins da Antártida
pinguins = sns.load_dataset("penguins").dropna().reset_index(drop=True)
print("Formato:", pinguins.shape)

# Correlação entre comprimento e profundidade do bico, ignorando a espécie
corr_geral = pinguins["bill_length_mm"].corr(pinguins["bill_depth_mm"])
print(f"\nCorrelação GERAL comprimento × profundidade do bico: {corr_geral:.2f}  (negativa!)")

# A mesma correlação, espécie por espécie
for especie, grupo in pinguins.groupby("species"):
    print(f"  dentro de {especie:10s}: {grupo['bill_length_mm'].corr(grupo['bill_depth_mm']):.2f}  (positiva!)")

In [ ]:
# O gráfico deixa o paradoxo evidente: uma reta geral descendente, três retas por espécie ascendentes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.regplot(data=pinguins, x="bill_length_mm", y="bill_depth_mm", scatter_kws={"alpha": 0.4}, color="gray", ax=axes[0])
axes[0].set_title(f"Todos juntos: correlação {corr_geral:.2f}")
for especie, grupo in pinguins.groupby("species"):
    sns.regplot(data=grupo, x="bill_length_mm", y="bill_depth_mm", scatter_kws={"alpha": 0.4}, label=especie, ax=axes[1])
axes[1].legend(); axes[1].set_title("Separado por espécie: correlações positivas")
plt.tight_layout(); plt.show()

print("Interpretação: 'bicos mais longos são mais finos' é FALSO dentro de qualquer espécie.")
print("A conclusão errada nasce de misturar grupos diferentes. Sempre pergunte: existe uma variável de grupo escondida?")

## Fase 5 (versão EDA) — Checklist de qualidade

Antes de entregar, o notebook precisa passar em testes automáticos. Se alguma verificação falhar, a célula quebra, e é isso que queremos: descobrir agora, não na apresentação.

In [ ]:
# Cada assert é um item do checklist. Uma falha aqui interrompe a execução com a mensagem explicativa.
assert df.duplicated().sum() == 0,                 "❌ ainda há duplicatas"
assert df.isna().sum().sum() == 0,                 "❌ ainda há valores faltantes"
assert df["tarifa"].dtype == "float64",            "❌ tarifa não é numérica"
assert df["idade"].between(0, 100).all(),          "❌ idade fora do intervalo plausível"
assert (df["tarifa"] >= 0).all(),                  "❌ tarifa negativa"
assert set(df["sexo"].cat.categories) == {"female", "male"}, "❌ categorias de sexo inconsistentes"
assert len(decisoes) >= 8,                         "❌ registro de decisões incompleto"

print("✅ Todas as verificações passaram.")
print(f"✅ {len(df)} linhas limpas, {df.shape[1]} colunas, {len(decisoes)} decisões documentadas.")
print("\nÚltimo passo manual: menu Kernel → 'Restart & Run All'. Se rodar do zero sem erro, o notebook é auditável.")

In [ ]:
# Síntese para o relatório: as 3 evidências que respondem à pergunta central da Fase 1
sintese = pd.DataFrame([
    ("Sexo",   f"{df.groupby('sexo', observed=True)['sobreviveu'].mean().diff().iloc[-1]:+.0%} de diferença (mulheres vs homens)", "forte, consistente em todas as classes"),
    ("Classe", f"1ª: {taxa.loc[1].mean():.0%} · 3ª: {taxa.loc[3].mean():.0%} (média entre sexos)", "forte, mesmo dentro de cada sexo"),
    ("Tarifa", f"Spearman {spearman.loc['tarifa','sobreviveu']:.2f} geral, mas quase zero dentro de cada classe", "NÃO tem efeito próprio: é proxy da classe"),
], columns=["fator", "evidencia", "leitura"])

sintese

## Exercícios guiados

### Exercício 1 — Porto de embarque e sobrevivência

A companhia quer saber se o **porto de embarque** (C, Q, S) influenciou a sobrevivência. Calcule a taxa por porto, depois cruze com a classe e escreva UMA frase dizendo se o porto tem efeito próprio ou se é outra variável de confusão, como a tarifa foi.

In [ ]:
# Esqueleto do Exercício 1
# TODO: taxa de sobrevivência por porto de embarque (groupby + mean), ordenada da maior para a menor
taxa_porto = ...

# TODO: proporção de passageiros de cada classe em cada porto (pivot_table com aggfunc='size' e normalize por linha)
classe_por_porto = ...

# TODO: taxa de sobrevivência por porto DENTRO de cada classe (pivot_table index=embarque, columns=classe)
taxa_porto_classe = ...

# TODO: complete aqui a frase de interpretação
print("Interpretação: ...")

💡 **Dica:** `pd.crosstab(df["embarque"], df["classe"], normalize="index")` dá a composição de classes por porto. Compare com a tabela de taxas dentro de cada classe: se as linhas ficam parecidas, o porto não tem efeito próprio.

In [ ]:
# ✅ SOLUÇÃO — Exercício 1
taxa_porto = df.groupby("embarque", observed=True)["sobreviveu"].mean().sort_values(ascending=False).round(2)
print("Taxa de sobrevivência por porto:")
print(taxa_porto, "\n")

# Composição de classes em cada porto: Cherbourg embarcou muito mais 1ª classe
classe_por_porto = pd.crosstab(df["embarque"], df["classe"], normalize="index").round(2)
print("Proporção de cada classe por porto:")
print(classe_por_porto, "\n")

# Dentro de cada classe, a diferença entre portos encolhe
taxa_porto_classe = df.pivot_table(index="embarque", columns="classe", values="sobreviveu", aggfunc="mean", observed=True).round(2)
print("Taxa por porto DENTRO de cada classe:")
print(taxa_porto_classe)

print("\nInterpretação: Cherbourg (C) tem a maior taxa porque metade dos seus passageiros era de 1ª classe;")
print("controlando por classe, o porto quase não faz diferença. É a mesma armadilha da tarifa: variável de confusão.")

### Exercício 2 — Mini-pipeline completo nos pinguins

Uma estação de pesquisa quer saber **se pinguins com nadadeiras maiores são mais pesados** e se isso vale para as três espécies. Recarregue `penguins` SEM o `dropna()`, diagnostique os faltantes, decida e registre o tratamento, e responda à pergunta com um número por espécie e um gráfico com frase de interpretação.

In [ ]:
# Esqueleto do Exercício 2
decisoes_ex2 = []
pinguins_bruto = sns.load_dataset("penguins")

# TODO: diagnóstico: quantos faltantes por coluna e quantas linhas têm QUALQUER faltante
...

# TODO: decida: remover as linhas ou imputar? Registre em decisoes_ex2 (problema, decisao, justificativa)
pinguins_limpo = ...

# TODO: correlação flipper_length_mm × body_mass_g geral e por espécie
...

# TODO: gráfico de dispersão colorido por espécie + print de interpretação
...

💡 **Dica:** conte quantas linhas seriam perdidas com `dropna()` e compare com o total. Perder 3% de linhas sem padrão de grupo é bem diferente de perder 20% concentrados numa espécie. Verifique isso com `isna().any(axis=1)` cruzado com `species`.

In [ ]:
# ✅ SOLUÇÃO — Exercício 2
pinguins_bruto = sns.load_dataset("penguins")
decisoes_ex2 = []

# Diagnóstico: poucos faltantes, e a falta não se concentra em uma espécie
print("Faltantes por coluna:\n", pinguins_bruto.isna().sum(), "\n")
linhas_com_falta = pinguins_bruto.isna().any(axis=1)
print(f"Linhas com algum faltante: {linhas_com_falta.sum()} de {len(pinguins_bruto)} ({linhas_com_falta.mean():.1%})")
print("Por espécie:\n", pinguins_bruto.loc[linhas_com_falta, "species"].value_counts(), "\n")

# Decisão: remover, porque a perda é pequena (~3%) e espalhada entre espécies (sem viés de grupo)
pinguins_limpo = pinguins_bruto.dropna().reset_index(drop=True)
decisoes_ex2.append({"problema": "11 linhas com faltantes (3%)", "decisao": "remover linhas",
                     "justificativa": "perda pequena e distribuída entre as espécies; imputar 'sex' seria chute"})

# Correlação geral e por espécie: aqui NÃO há paradoxo, o sinal se mantém
print(f"Correlação geral nadadeira × massa: {pinguins_limpo['flipper_length_mm'].corr(pinguins_limpo['body_mass_g']):.2f}")
for especie, grupo in pinguins_limpo.groupby("species"):
    print(f"  {especie:10s}: {grupo['flipper_length_mm'].corr(grupo['body_mass_g']):.2f}  (n = {len(grupo)})")

In [ ]:
# ✅ SOLUÇÃO — Exercício 2 (continuação): gráfico + interpretação + registro
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=pinguins_limpo, x="flipper_length_mm", y="body_mass_g", hue="species", alpha=0.7, ax=ax)
ax.set_title("Nadadeira × massa corporal por espécie")
plt.tight_layout(); plt.show()

print("Interpretação: nadadeiras maiores acompanham maior massa em TODAS as espécies (correlações entre 0,5 e 0,9);")
print("a espécie Gentoo é maior nas duas medidas, mas a relação vale dentro de cada grupo, sem inversão de sinal.")
print("\nRegistro de decisões do exercício:")
print(pd.DataFrame(decisoes_ex2).to_string(index=False))

## Desafio final (em equipe, sem solução)

Uma montadora quer entender **o que determina o consumo de combustível** de um carro. Use o dataset público `sns.load_dataset("mpg")` (398 carros, com faltantes em `horsepower`) e percorra as fases 1 a 3 do CRISP-DM mais a EDA, exatamente como fizemos aqui.

**Entrega:** um notebook novo, executável do zero, com:
1. Ficha do projeto (Fase 1) e tabela de perguntas de negócio → métricas nos dados.
2. Dicionário de dados com tipo esperado × tipo real.
3. Registro de decisões com pelo menos 4 entradas justificadas (faltantes, outliers, tipos, categorias como `origin`).
4. EDA com no mínimo: um histograma, um boxplot por grupo, uma matriz de correlação e uma dispersão, cada um com frase de interpretação.
5. Um teste explícito de variável de confusão (ex.: `weight` explica a relação entre `cylinders` e `mpg`?) e um comentário sobre viés de amostragem (de que anos e origens são os carros?).
6. Checklist final com `assert`s que passem.

**Critérios de avaliação:** roda do zero sem erro (25%); cada decisão de preparação tem justificativa escrita (25%); cada gráfico tem interpretação e nenhuma conclusão vai além do que os dados sustentam (30%); o teste de confusão está correto (20%).

Os notebooks das equipes serão o material de trabalho da síntese na Aula 06.

## Encerramento

**O que praticamos hoje**
- Fase 1: ficha do projeto e tradução de perguntas de negócio em métricas.
- Fase 2: dicionário de dados, diagnóstico de faltantes com padrão (viés) e mapa visual.
- Fase 3: tipos, categorias, duplicatas, outliers e imputação, nesta ordem e com registro de decisões.
- EDA: histogramas (com escala log), boxplots por grupo, Pearson × Spearman, dispersão.
- Três alertas de Grus na prática: confusão pela classe, viés pelos faltantes, Paradoxo de Simpson nos pinguins.
- Checklist com `assert` e "Restart & Run All" como prova de reprodutibilidade.

**Próxima aula (Aula 06):** síntese dos notebooks das equipes e ponte para a Fase 4, Modelagem, onde entram os primeiros algoritmos de aprendizado de máquina.